# Улучшение качества OSNet — variant 20 / Run All

Это **новые исследовательские эксперименты**, а не проверка Docker/устройств. Диагностика ошибок → обучаемая визуальная проверка пар → эксперименты с графиком обучения → автоматическое подтверждение перспективных вариантов.

**MVP, release-integration, исходные bbox, validation и прежние runs не меняются.** Внешние данные, детекторы, OCR и обработка номерной зоны не используются. Outer/test не оцениваются. Для запуска нужен прежний исследовательский `.venv` из **Car-classification-MSK**, не `.venv` релизной копии.

Нажми **Restart Kernel → Run All**. Интернет не нужен. Оставь компьютер подключённым к питанию и без сна. Общего ограничения времени нет.


In [1]:
from pathlib import Path
import os
import sys
import json

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/quality_experiment.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
from IPython.display import Markdown, display
from training import quality_experiment as experiment

SOURCE_RUN = 'review_v1'
POLICY_RUN = 'policy_v1'
RUN_NAME = 'quality_v1'
RUN_HEAD = True
RUN_CLOCK = True
AUTO_CONFIRM = True

source_manifest = json.loads((REPO / 'OSNet-AIN-x1.0/variant_16_review_protocol/runs' / SOURCE_RUN / 'manifest.json').read_text())
torch.set_num_threads(source_manifest['runtime']['torch_threads'])
torch.use_deterministic_algorithms(source_manifest['runtime']['deterministic'])
print('Repository:', REPO)
print('Kernel:', sys.executable)
print('Device from frozen source:', source_manifest['runtime']['device'])
print('Run:', RUN_NAME, '| head:', RUN_HEAD, '| clock:', RUN_CLOCK, '| confirmation:', AUTO_CONFIRM)


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Device from frozen source: mps
Run: quality_v1 | head: True | clock: True | confirmation: True


## 1. Быстрые проверки кода

Тесты используют синтетические данные и временные каталоги. Проверяют границы train/holdout, junk, независимость pair-score, возобновление, BN и правила отбора. Никакого полного обучения на этом этапе нет.


In [2]:
import subprocess
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_quality_experiment.py', 'tests/test_quality_notebook.py'], check=True)


..............................                                           [100%]
30 passed in 2.99s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_quality_experiment.py', 'tests/test_quality_notebook.py'], returncode=0)

## 2. Preflight и замораживание эксперимента

Читаются завершённые v16/v18: проверяются исходные данные, разбиения, веса и контрольные суммы. Используется то же устройство/окружение, что в исходном эксперименте — переезд на GPU не требуется.

`RUN_NAME` неизменён: повторный Run All продолжит незавершённые задачи; готовые результаты проверятся по checksum и не пересчитаются. При изменении кода/настроек нужен новый RUN_NAME. Не запускай два обучения параллельно.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN, policy_run=POLICY_RUN,
                             run_head=RUN_HEAD, run_clock=RUN_CLOCK, auto_confirm=AUTO_CONFIRM)
print('Результаты:', context['output'])
print('Seed:', context['seeds'])
display(Markdown('### Замороженный план'))
print(json.dumps(context['manifest']['quality_plan'], ensure_ascii=False, indent=2))


Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_20_quality/runs/quality_v1
Seed: (20260915, 20260916, 20260917)


### Замороженный план

{
  "run_head": true,
  "run_clock": true,
  "auto_confirm": true,
  "head": {
    "images_per_identity": 4,
    "pool": 50,
    "positives_per_query": 2,
    "negatives_per_query": 8,
    "hidden": 64,
    "epochs": 12,
    "batch_size": 256,
    "learning_rate": 0.001,
    "weight_decay": 0.0001
  },
  "head_weights": {
    "baseline": 0.0,
    "head_w05": 0.05,
    "head_w10": 0.1
  },
  "clock_jobs": {
    "R1_control": {
      "name": "R1_control",
      "base_variant": "R1_resolution256",
      "ce_scale": 1.0,
      "relation_weight": 0.0,
      "stop_step": 800,
      "lr_horizon": 1700
    },
    "R1_cosine800": {
      "name": "R1_cosine800",
      "base_variant": "R1_resolution256",
      "ce_scale": 1.0,
      "relation_weight": 0.0,
      "stop_step": 800,
      "lr_horizon": 800
    },
    "R1_full1700": {
      "name": "R1_full1700",
      "base_variant": "R1_resolution256",
      "ce_scale": 1.0,
      "relation_weight": 0.0,
      "stop_step": 1700,
      "lr_horizon":

## 3. Вся очередь до завершения

**Диагностика:** исходный R1 на primary, пропуски нужной машины в top-50, запас перестановки, улучшения/ухудшения от графа. Сохраняются HTML с реальными crop.

**Парная модель:** frozen OSNet conv5 → локальные соответствия → маленькая обучаемая голова. Только train-ID данного encoder; positives cross-camera, negatives из реального top-50. Проверка на других identity, которых не видел ни encoder, ни head. Это не OOF и не калиброванный confidence: меняется только ranking. Два веса добавки — 0.05/0.10, baseline обязателен.

**Обучение OSNet:** контроль 800 шагов / horizon 1700; 800/800; 1700/1700. Для полного цикла дополнительно BN-only и среднее checkpoint 1400/1600/1700 с тем же пересчётом BN на train. Это усреднение checkpoint, **не EMA**. Новый sampler/loss не добавляется.

Пилоты часов обучения сначала на одном seed; только выигравший вариант с контролями подтверждается на трёх seed и alternate. У головы primary сразу на трёх seed; alternate только для замороженного победителя. Outer, новые full-train веса и переключение MVP здесь не запускаются.

Прогресс: `[START]/[DONE]`, эпохи головы, шаги encoder, loss, elapsed/ETA. Обычные P×K batch сохранены; блоки по 200 шагов нужны для сохранения состояния, не заменяют batch. Ошибки независимой задачи сохраняются, остальные задачи продолжаются. Ошибка целостности останавливает всё.


In [4]:
result = experiment.run(context)
print('Статус:', result['status'])
print('Сохранность защищённых файлов:', result.get('protected_unchanged'))
print('Длительность, часов:', round(result['elapsed_seconds'] / 3600, 2))


[START] diagnostics_primary | elapsed 0.00 h
  features primary/20260915/evaluation: 32/1036
  features primary/20260915/evaluation: 352/1036
  features primary/20260915/evaluation: 672/1036
  features primary/20260915/evaluation: 992/1036
  features primary/20260915/evaluation: 1036/1036
  diagnostics regular_20260915: {'known_queries': 148, 'no_valid_positive_in_top50': 6, 'reordering_headroom': 33, 'graph_hurt_vs_raw': 12, 'graph_helped_vs_raw': 22}
  diagnostics regular_20261016: {'known_queries': 148, 'no_valid_positive_in_top50': 3, 'reordering_headroom': 31, 'graph_hurt_vs_raw': 8, 'graph_helped_vs_raw': 18}
  diagnostics regular_20261117: {'known_queries': 148, 'no_valid_positive_in_top50': 4, 'reordering_headroom': 38, 'graph_hurt_vs_raw': 9, 'graph_helped_vs_raw': 22}
[DONE] diagnostics_primary | 0.4 min
[START] head_train_primary_20260915 | elapsed 0.01 h
  features primary/20260915/train: 32/2960
  features primary/20260915/train: 352/2960
  features primary/20260915/train:

## 4. Итоговый отчёт

Рост train accuracy/loss и oracle не считаются улучшением поиска. Смотрим **парную разницу mAP**, три seed, фиксированный выбор на alternate и число исправленных/испорченных запросов. Исторические 0.8147 и inner ~0.84 нельзя сравнивать как одну выборку. Даже успешное подтверждение не означает автоматическую победу над MVP на закрытом тесте.

C/F1/TNR для нового encoder сохраняются как отдельная внутренняя диагностика с раздельными identity калибровки и оценки. Ни эти пороги, ни sigmoid парной головы не заменяют frozen-пороги рабочего приложения.

Контроль остаточного сигнала номерной зоны этим notebook не решён: не делаем заявления о его отсутствии. При положительном результате нужен отдельный план финального обучения/оценки и проверка допустимости признаков.


In [5]:
report = context['output'] / 'REPORT.md'
display(Markdown(report.read_text()))
print('Полные результаты:', context['output'] / 'results.json')
print('Визуальная диагностика:')
for p in sorted((context['output'] / 'tasks/diagnostics_primary').glob('errors_*.html')):
    print(p)
if result['status'] != 'complete':
    raise RuntimeError('Есть незавершённые/ошибочные задачи. См. REPORT.md и tasks/*/error.json; после исправления повтори тот же RUN_NAME без изменения рецепта.')


# v20: эксперименты на качество

Статус: **complete**.
Нового outer/test mAP нет. MVP, исходная validation и bbox не изменялись.
Пилот/подтверждение — внутренние сравнения, не автоматическое продвижение.

## Задачи

| Задача | Статус |
|---|---|
| diagnostics_primary | complete |
| head_train_primary_20260915 | complete |
| head_eval_primary_20260915 | complete |
| head_train_primary_20260916 | complete |
| head_eval_primary_20260916 | complete |
| head_train_primary_20260917 | complete |
| head_eval_primary_20260917 | complete |
| head_selection | complete |
| clock_train_primary_R1_control_20260915 | complete |
| clock_eval_primary_R1_control_20260915 | complete |
| clock_train_primary_R1_cosine800_20260915 | complete |
| clock_eval_primary_R1_cosine800_20260915 | complete |
| clock_train_primary_R1_full1700_20260915 | complete |
| clock_eval_primary_R1_full1700_20260915 | complete |
| clock_eval_primary_R1_full1700_bn_20260915 | complete |
| clock_eval_primary_R1_full1700_avg_20260915 | complete |
| clock_selection | complete |
| clock_train_primary_R1_control_20260916 | complete |
| clock_eval_primary_R1_control_20260916 | complete |
| clock_train_primary_R1_control_20260917 | complete |
| clock_eval_primary_R1_control_20260917 | complete |
| clock_train_primary_R1_full1700_20260916 | complete |
| clock_eval_primary_R1_full1700_bn_20260916 | complete |
| clock_train_primary_R1_full1700_20260917 | complete |
| clock_eval_primary_R1_full1700_bn_20260917 | complete |

## head

Выбор: `baseline`

| Разбиение | Вариант | Seed | mAP@10 |
|---|---|---:|---:|
| primary | baseline | 20260915 | 0.84410 |
| primary | baseline | 20260916 | 0.84836 |
| primary | baseline | 20260917 | 0.83637 |
| primary | head_w05 | 20260915 | 0.84324 |
| primary | head_w05 | 20260916 | 0.84873 |
| primary | head_w05 | 20260917 | 0.83597 |
| primary | head_w10 | 20260915 | 0.84282 |
| primary | head_w10 | 20260916 | 0.84935 |
| primary | head_w10 | 20260917 | 0.83532 |

Primary/alternate gates и условный bootstrap: см. results.json.

## clock

Выбор: `R1_full1700_bn`

| Разбиение | Вариант | Seed | mAP@10 |
|---|---|---:|---:|
| primary pilot | R1_control | 20260915 | 0.84410 |
| primary pilot | R1_cosine800 | 20260915 | 0.83753 |
| primary pilot | R1_full1700 | 20260915 | 0.84003 |
| primary pilot | R1_full1700_bn | 20260915 | 0.84842 |
| primary pilot | R1_full1700_avg | 20260915 | 0.84932 |
| primary | R1_control | 20260915 | 0.84410 |
| primary | R1_control | 20260916 | 0.84836 |
| primary | R1_control | 20260917 | 0.83637 |
| primary | R1_full1700_bn | 20260915 | 0.84842 |
| primary | R1_full1700_bn | 20260916 | 0.84162 |
| primary | R1_full1700_bn | 20260917 | 0.82466 |

Primary/alternate gates и условный bootstrap: см. results.json.

## Как читать результат

- Сначала смотреть прирост относительно парного R1-контроля, затем подтверждение на alternate.
- Три seed и повторные query-эпизоды не являются новыми независимыми identity.
- head меняет только ranking; raw-кандидат/отказы не меняются при любом фиксированном пороге.
- Clock C/F1/TNR: отдельная внутренняя диагностика с раздельными identity калибровки/оценки.
- errors_*.html в head_eval содержат реальные crop для визуального разбора. Oracle — только верхняя граница.
- Усреднение сравнивается также с BN-only контролем; параметры не выбираются по outer.
- Номерная зона специально не используется/не обрабатывается; отсутствие остаточного сигнала не доказано.
- Автоматического объединения head и нового encoder, full-train, ONNX-релиза и смены MVP нет.
- Сохранность источников/изображений: True.


Полные результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_20_quality/runs/quality_v1/results.json
Визуальная диагностика:
/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_20_quality/runs/quality_v1/tasks/diagnostics_primary/errors_regular_20260915.html
/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_20_quality/runs/quality_v1/tasks/diagnostics_primary/errors_regular_20261016.html
/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_20_quality/runs/quality_v1/tasks/diagnostics_primary/errors_regular_20261117.html
